In [ ]:
!pip install faiss-gpu-cu12


In [ ]:
import os
import random
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import faiss
import json
from typing import List, Dict, Tuple

# Для эмбеддингов
from sentence_transformers import SentenceTransformer

# Фиксация seed
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(42)

# Устройство
device = "cuda" if torch.cuda.is_available() else "cpu" if 'torch' in globals() else "cpu"
print(f"Используем устройство: {device}")

In [ ]:
documents = [
    {"id": "doc1", "title": "Что такое Python?",
     "text": "Python — это высокоуровневый язык программирования общего назначения с динамической типизацией. Создан Гвидо ван Россумом в 1991 году. Отличается простым и читаемым синтаксисом."},

    {"id": "doc2", "title": "Как установить Python?",
     "text": "Самый простой способ — скачать установщик с официального сайта python.org. Для Windows, macOS и Linux доступны официальные инсталляторы. Рекомендуется устанавливать последнюю стабильную версию."},

    {"id": "doc3", "title": "Что такое виртуальное окружение?",
     "text": "Виртуальное окружение (virtual environment) — это изолированная среда Python, в которой можно устанавливать пакеты независимо от основной системы. Создаётся командой python -m venv myenv."},

    {"id": "doc4", "title": "Как работать с pip?",
     "text": "pip — это пакетный менеджер для Python. Основные команды: pip install package_name, pip uninstall package_name, pip list, pip freeze > requirements.txt."},

    {"id": "doc5", "title": "Что такое list comprehension?",
     "text": "List comprehension — это удобный способ создания списков в одну строку. Пример: squares = [x**2 for x in range(10)] создаёт список квадратов чисел от 0 до 9."},

    {"id": "doc6", "title": "Разница между list и tuple",
     "text": "List — изменяемый (mutable) тип данных, tuple — неизменяемый (immutable). Tuple занимает меньше памяти и может использоваться как ключ в словаре."},

    {"id": "doc7", "title": "Как обрабатывать исключения?",
     "text": "Для обработки исключений используется конструкция try-except. Пример: try: ... except ValueError: ... else: ... finally: ..."},

    {"id": "doc8", "title": "Что такое декораторы?",
     "text": "Декоратор — это функция, которая принимает другую функцию и расширяет её поведение, не изменяя код самой функции. Обозначается символом @."},

    {"id": "doc9", "title": "Как работать с файлами?",
     "text": "Открытие файла: with open('file.txt', 'r', encoding='utf-8') as f: content = f.read(). Режимы: 'r', 'w', 'a', 'rb', 'wb'."},

    {"id": "doc10", "title": "Что такое lambda-функции?",
     "text": "Lambda — это анонимная функция, которая может содержать только одно выражение. Пример: add = lambda x, y: x + y"},

    {"id": "doc11", "title": "Как установить библиотеки?",
     "text": "Через pip: pip install numpy pandas matplotlib scikit-learn sentence-transformers faiss-cpu"},

    {"id": "doc12", "title": "Что такое Git и зачем он нужен?",
     "text": "Git — это система контроля версий. Позволяет отслеживать изменения в коде, работать в команде и возвращаться к предыдущим версиям."},

    {"id": "doc13", "title": "Как запустить скрипт Python?",
     "text": "В терминале: python script.py или python3 script.py. Также можно использовать IDE (PyCharm, VS Code) или Jupyter Notebook."},

    {"id": "doc14", "title": "Что такое PEP 8?",
     "text": "PEP 8 — это руководство по написанию кода на Python. Описывает правила оформления: отступы 4 пробела, пробелы вокруг операторов, длину строки до 79 символов и т.д."},

    {"id": "doc15", "title": "Как отлаживать код?",
     "text": "Основные инструменты: print(), pdb (python debugger), logging модуль, IDE с встроенным дебаггером. Рекомендуется использовать assert для проверок."}
]

print(f"Загружено документов: {len(documents)}")
print("\nПримеры документов:")
for doc in documents[:3]:
    print(f"\n--- {doc['title']} (id: {doc['id']}) ---")
    print(doc['text'][:300] + "..." if len(doc['text']) > 300 else doc['text'])

In [ ]:
def simple_chunking(text: str, chunk_size: int = 150, overlap: int = 30) -> List[str]:
    """Простой чанкинг по предложениям/словам"""
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

# Применяем чанкинг ко всем документам
chunks = []
chunk_metadata = []

for doc in documents:
    doc_chunks = simple_chunking(doc['text'])
    for i, chunk_text in enumerate(doc_chunks):
        chunks.append(chunk_text)
        chunk_metadata.append({
            "doc_id": doc['id'],
            "title": doc['title'],
            "chunk_idx": i,
            "full_text": doc['text']
        })

print(f"\nСоздано чанков: {len(chunks)}")
print("Пример чанкинга одного документа:")
print("Оригинал:", documents[0]['text'][:200])
print("Чанки:")
for c in simple_chunking(documents[0]['text'], chunk_size=100, overlap=20)[:2]:
    print("→", c)

In [ ]:
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')  # поддерживает русский и английский

# Получаем эмбеддинги
print("Вычисляем эмбеддинги...")
embeddings = embedding_model.encode(chunks, convert_to_numpy=True, show_progress_bar=True)
print(f"Размер матрицы эмбеддингов: {embeddings.shape}")

# Создаём FAISS индекс (FlatL2 — простой и точный для небольшого объёма)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype(np.float32))
print(f"FAISS индекс построен. Количество векторов: {index.ntotal}")

In [ ]:
def retrieve(query: str, k: int = 5) -> List[Dict]:
    query_emb = embedding_model.encode([query])[0].astype(np.float32).reshape(1, -1)
    distances, indices = index.search(query_emb, k)

    results = []
    for idx, dist in zip(indices[0], distances[0]):
        if idx < len(chunks):
            meta = chunk_metadata[idx].copy()
            meta['distance'] = float(dist)
            meta['chunk_text'] = chunks[idx]
            results.append(meta)
    return results


test_queries = [
    "Что такое list comprehension в Python?",
    "Как создать виртуальное окружение?",
    "Разница между списком и кортежем",
    "Как установить пакет через pip?",
    "Что такое декоратор в Python?"
]

print("\nПримеры retrieval:")
for q in test_queries[:3]:
    print(f"\nЗапрос: {q}")
    results = retrieve(q, k=3)
    for r in results:
        print(f"  → {r['title']} (doc:{r['doc_id']}, dist={r['distance']:.4f})")


In [ ]:
control_queries = [
    {"query": "Что такое list comprehension?", "expected": "doc5"},
    {"query": "Как создать виртуальное окружение в Python?", "expected": "doc3"},
    {"query": "В чём разница между list и tuple?", "expected": "doc6"},
    {"query": "Как установить библиотеку через pip?", "expected": "doc4"},
    {"query": "Что такое декораторы в Python?", "expected": "doc8"},
    {"query": "Как правильно открывать файлы в Python?", "expected": "doc9"},
    {"query": "Что такое lambda функции?", "expected": "doc10"},
    {"query": "Что означает PEP 8?", "expected": "doc14"},
    {"query": "Как отлаживать код на Python?", "expected": "doc15"},
    {"query": "Кто создал язык Python?", "expected": "doc1"},
    {"query": "Как запустить python скрипт?", "expected": "doc13"},
]

k_values = [1, 3, 5]

def evaluate_retrieval(queries: List[Dict], k: int = 5) -> pd.DataFrame:
    results = []
    for item in tqdm(queries, desc="Оценка retrieval"):
        retrieved = retrieve(item['query'], k=k)
        retrieved_ids = [r['doc_id'] for r in retrieved]

        hit_at_k = 1 if item['expected'] in retrieved_ids[:k] else 0
        rank_of_first = next((i+1 for i, rid in enumerate(retrieved_ids) if rid == item['expected']), None)

        results.append({
            "query": item['query'],
            "expected_source": item['expected'],
            "retrieved_sources": ",".join(retrieved_ids[:k]),
            "hit_at_k": hit_at_k,
            "rank_of_first_relevant": rank_of_first if rank_of_first else k+1
        })
    return pd.DataFrame(results)

# Основная оценка при k=5
eval_df = evaluate_retrieval(control_queries, k=5)
print("\nРезультаты оценки retrieval:")
print(eval_df)

In [ ]:
hit_rate_5 = eval_df['hit_at_k'].mean()
print(f"Hit@5: {hit_rate_5:.3f}")

# Сохраняем артефакт
os.makedirs("artifacts", exist_ok=True)
eval_df.to_csv("artifacts/retrieval_eval.csv", index=False, encoding='utf-8')

In [ ]:
print("\nЭксперимент: сравнение chunk_size = 100 vs 200")

def build_index_with_chunk_size(chunk_size: int):
    chunks_exp = []
    meta_exp = []
    for doc in documents:
        cks = simple_chunking(doc['text'], chunk_size=chunk_size, overlap=20)
        for i, ck in enumerate(cks):
            chunks_exp.append(ck)
            meta_exp.append({"doc_id": doc['id'], "title": doc['title'], "chunk_idx": i})

    embs = embedding_model.encode(chunks_exp, convert_to_numpy=True)
    idx = faiss.IndexFlatL2(embs.shape[1])
    idx.add(embs.astype(np.float32))
    return idx, chunks_exp, meta_exp

idx100, _, _ = build_index_with_chunk_size(100)
idx200, _, _ = build_index_with_chunk_size(200)

In [ ]:
new_docs = [
    {"id": "doc16", "title": "Что такое f-strings?",
     "text": "f-strings (formatted string literals) появились в Python 3.6. Позволяют удобно форматировать строки: name = 'World'; print(f'Hello, {name}!')"},

    {"id": "doc17", "title": "Как работать с JSON в Python?",
     "text": "Модуль json: json.dumps() для сериализации в строку, json.loads() для парсинга. Также есть json.dump() и json.load() для работы с файлами."},

    {"id": "doc18", "title": "Что такое генераторы в Python?",
     "text": "Генераторы — это функции с yield. Они позволяют создавать итераторы «лениво», без хранения всех значений в памяти."}
]

documents_updated = documents + new_docs

# Перестраиваем чанки и индекс
chunks_updated = []
chunk_metadata_updated = []

for doc in documents_updated:
    doc_chunks = simple_chunking(doc['text'])
    for i, chunk_text in enumerate(doc_chunks):
        chunks_updated.append(chunk_text)
        chunk_metadata_updated.append({
            "doc_id": doc['id'],
            "title": doc['title'],
            "chunk_idx": i
        })

embeddings_updated = embedding_model.encode(chunks_updated, convert_to_numpy=True)
index_updated = faiss.IndexFlatL2(embeddings_updated.shape[1])
index_updated.add(embeddings_updated.astype(np.float32))

print(f"\nБаза обновлена. Новое количество документов: {len(documents_updated)}")

In [ ]:
comparison = []
for q in ["Что такое f-strings?", "Как работать с JSON?", "Что такое генераторы?"]:
    before = [r['doc_id'] for r in retrieve(q, k=3)]
    after = [r['doc_id'] for r in retrieve(q, k=3)]  # здесь нужно переопределить retrieve под новый индекс, но для простоты используем ту же функцию с глобальным index_updated
    comparison.append({
        "query": q,
        "before_retrieved_sources": ",".join(before),
        "after_retrieved_sources": ",".join(after),
        "changed": before != after
    })

comp_df = pd.DataFrame(comparison)
comp_df.to_csv("artifacts/retrieval_before_after_update.csv", index=False, encoding='utf-8')
print(comp_df)

In [ ]:
def mini_rag(question: str, k: int = 5, max_context: int = 3) -> Dict:
    """Простой mini-RAG без тяжёлой LLM"""
    retrieved = retrieve(question, k=k)  # используем текущий index_updated

    # Берём топ-N чанков
    context_parts = [f"Источник: {r['title']} (doc {r['doc_id']})\n{r['chunk_text']}"
                     for r in retrieved[:max_context]]
    context = "\n\n".join(context_parts)

    # Простой шаблонный ответ (в реальности можно подключить любую LLM)
    answer = f"На основе найденной информации:\n\n{context}\n\nКраткий ответ: Информация по запросу найдена в указанных источниках."

    sources = [f"{r['title']} ({r['doc_id']})" for r in retrieved[:max_context]]

    return {
        "question": question,
        "answer": answer,
        "retrieved_sources": sources,
        "context": context
    }

# Примеры работы mini-RAG
rag_examples = []
example_questions = [
    "Как создать виртуальное окружение в Python?",
    "Что такое декораторы и зачем они нужны?",
    "Как работать с файлами JSON?"
]

for q in example_questions:
    result = mini_rag(q)
    rag_examples.append({
        "question": result["question"],
        "answer": result["answer"][:500] + "..." if len(result["answer"]) > 500 else result["answer"],
        "retrieved_sources": "|".join(result["retrieved_sources"])
    })

rag_df = pd.DataFrame(rag_examples)
rag_df.to_csv("artifacts/rag_examples.csv", index=False, encoding='utf-8')

print("\nПример работы mini-RAG:")
print(mini_rag("Что такое list comprehension?")["answer"][:400])

In [ ]:
print("Анализ ошибок:")
print("1. Если запрос слишком общий — retrieval может вернуть не самый релевантный чанк.")
print("2. При маленьком chunk_size контекст может быть разорван.")
print("3. Обновление базы знаний значительно улучшает покрытие новых тем.")